# 6 — RNN training (V1, V2 and V3)

In this notebook, we train **three bidirectional RNN models** using our symptom dataset. We compare their accuracy and save the best one.

All three use 192 hidden neurons and one RNN layer. We change regularization and the learning-rate scheduler to see which works best.

We use the **same frozen training and validation files** for a fair comparison. The test set is saved for the final evaluation.

## 1. Import libraries and choose settings

PyTorch trains our models. If CUDA is available, we use the GPU.

In [ ]:
import copy
import hashlib
import json
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
BATCH_SIZE = 64
MAX_LENGTH = 96
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

# Find the main project directory.
ROOT = Path.cwd().resolve()
while not (ROOT / 'app' / 'dataset').is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Please open this notebook inside your medical-diagnosis project.')
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'app' / 'dataset' / 'output' / 'training'
MODEL_DIR = ROOT / 'app' / 'model' / 'rnn_selected'
REPORT_DIR = DATA_DIR / 'rnn_selected'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Project:', ROOT)

## 2. Load our dataset

We have symptom descriptions and their condition labels. Each description can have more than one label. We also check that the dataset has not changed since our previous experiments.

In [ ]:
TRAIN_FILE = DATA_DIR / 'train.json'
VAL_FILE = DATA_DIR / 'validation.json'
train_rows = json.loads(TRAIN_FILE.read_text(encoding='utf-8'))
val_rows = json.loads(VAL_FILE.read_text(encoding='utf-8'))

train_texts = [row['symptoms_classical'] for row in train_rows]
val_texts = [row['symptoms_classical'] for row in val_rows]
assert set(train_texts).isdisjoint(val_texts), 'Train and validation texts overlap'

labels = MultiLabelBinarizer()
y_train = labels.fit_transform([row['labels'] for row in train_rows])
unknown = set(label for row in val_rows for label in row['labels']) - set(labels.classes_)
assert not unknown, f'Unknown validation labels: {unknown}'
y_val = labels.transform([row['labels'] for row in val_rows])
class_names = labels.classes_.tolist()
supported = y_val.sum(axis=0) > 0

# File fingerprints make sure we compare models using the same dataset.
data_hashes = {
    'train_json_sha256': hashlib.sha256(TRAIN_FILE.read_bytes()).hexdigest(),
    'validation_json_sha256': hashlib.sha256(VAL_FILE.read_bytes()).hexdigest(),
}
expected_hashes = {
    'train_json_sha256': '292fc80ddc16cc5afa9b49d9b93263775118e14af3b3de2795db7d06f336a6e1',
    'validation_json_sha256': 'c61e0a2ab6524829ddf29979de836c11623f40c1d3a10084af6a34cc0b340f19',
}
assert data_hashes == expected_hashes, 'Dataset changed. Check your train.json and validation.json.'

print('Train:', len(train_rows), '| Validation:', len(val_rows))
print('Conditions:', len(class_names), '| In validation:', int(supported.sum()))
print('Test set: NOT USED')

## 3. Convert symptom text into numbers

We build a vocabulary using only training text. Each symptom description becomes a list of word IDs, padded to 96 words.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?|\d+", text.lower())

counts = Counter(word for text in train_texts for word in tokenize(text))
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in counts.most_common(24998):
    vocab[word] = len(vocab)

def encode(text):
    words = tokenize(text)[:MAX_LENGTH]
    ids = [vocab.get(word, 1) for word in words] or [1]
    return ids + [0] * (MAX_LENGTH - len(ids))

train_x = torch.tensor([encode(text) for text in train_texts], dtype=torch.long)
val_x = torch.tensor([encode(text) for text in val_texts], dtype=torch.long)
train_y = torch.tensor(y_train, dtype=torch.float32)
val_y = torch.tensor(y_val, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=BATCH_SIZE)
print('Vocabulary size:', len(vocab))

## 4. Build the RNN model

The model uses word embeddings, a **bidirectional vanilla RNN** and a final layer that gives a score for each condition. We use the final hidden states from both directions.

In [ ]:
class SymptomRNN(nn.Module):
    def __init__(self, vocab_size, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.rnn = nn.RNN(128, 192, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(192 * 2, num_classes)

    def forward(self, x):
        lengths = (x != 0).sum(dim=1).cpu()
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths, batch_first=True,
                                      enforce_sorted=False)
        _, hidden = self.rnn(packed)
        sentence = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(sentence))

## 5. Our three versions

We use the same RNN architecture but change the training settings:

- **V1:** standard RNN settings.
- **V2:** learning-rate scheduler to reduce the learning rate when validation loss stops improving.
- **V3:** stronger dropout and weight decay to reduce overfitting.

In [ ]:
versions = {
    'v1': {'dropout': 0.30, 'lr': 0.001, 'weight_decay': 0.0001,
           'scheduler': False, 'epochs': 20, 'patience': 4},
    'v2': {'dropout': 0.30, 'lr': 0.001, 'weight_decay': 0.0001,
           'scheduler': True, 'epochs': 35, 'patience': 8},
    'v3': {'dropout': 0.45, 'lr': 0.001, 'weight_decay': 0.01,
           'scheduler': False, 'epochs': 30, 'patience': 5},
}
display(pd.DataFrame(versions).T)

## 6. Train each model

We use weighted binary cross-entropy for multi-label classification, clip gradients to keep training stable and stop early if validation loss stops improving. We save the weights from the best epoch.

In [ ]:
positive = y_train.sum(axis=0)
class_weights = np.clip((len(train_rows) - positive) / np.maximum(positive, 1), 1, 15)
loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
)

def train_model(settings):
    torch.manual_seed(SEED)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(SEED)

    model = SymptomRNN(len(vocab), len(class_names), settings['dropout']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['lr'],
                                  weight_decay=settings['weight_decay'])
    scheduler = None
    if settings['scheduler']:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-5)

    history = []
    best_loss = float('inf')
    best_epoch = 0
    waiting = 0
    start = time.perf_counter()

    for epoch in range(1, settings['epochs'] + 1):
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(x)

        model.eval()
        val_loss = 0
        all_scores = []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                val_loss += loss_fn(logits, y).item() * len(x)
                all_scores.append(torch.sigmoid(logits).cpu().numpy())

        val_loss /= len(val_rows)
        scores = np.concatenate(all_scores)
        epoch_f1 = f1_score(y_val, scores >= 0.5, average='micro', zero_division=0)
        top5 = np.argsort(-scores, axis=1)[:, :5]
        epoch_top5 = np.mean(np.take_along_axis(y_val, top5, axis=1).sum(axis=1) > 0)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss / len(train_rows),
            'validation_loss': val_loss,
            'validation_f1_at_05': float(epoch_f1),
            'validation_top5': float(epoch_top5),
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
        print(f"Epoch {epoch:02d}  val loss={val_loss:.4f}  F1={epoch_f1:.2%}  Top-5={epoch_top5:.2%}")

        if val_loss < best_loss - 0.0001:
            best_loss = val_loss
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())
            waiting = 0
        else:
            waiting += 1

        if scheduler is not None:
            scheduler.step(val_loss)
        if waiting >= settings['patience']:
            print('Early stopping')
            break

    model.load_state_dict(best_weights)
    return model, pd.DataFrame(history), best_epoch, time.perf_counter() - start

## 7. Evaluate on validation data

We find the threshold with the best **Micro F1** on validation and calculate precision, recall, Macro F1 and **Top-1, Top-3, Top-5**.

Top-K is counted as a hit if at least one true condition appears in the top K. These results are from NHS-derived examples, not real patient diagnoses.

In [ ]:
def evaluate_model(model):
    model.eval()
    all_scores = []
    with torch.no_grad():
        for x, _ in val_loader:
            all_scores.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
    scores = np.concatenate(all_scores)

    thresholds = [0.005, 0.01, 0.02, 0.025] + list(np.round(np.arange(0.05, 0.951, 0.025), 3))
    threshold_search = []
    for t in thresholds:
        predicted = (scores >= t).astype(int)
        threshold_search.append({
            'threshold': float(t),
            'micro_precision': float(precision_score(y_val, predicted, average='micro', zero_division=0)),
            'micro_recall': float(recall_score(y_val, predicted, average='micro', zero_division=0)),
            'micro_f1': float(f1_score(y_val, predicted, average='micro', zero_division=0)),
        })

    best = max(threshold_search, key=lambda row: row['micro_f1'])
    predicted = (scores >= best['threshold']).astype(int)
    top_k = {}
    for k in (1, 3, 5):
        indexes = np.argsort(-scores, axis=1)[:, :k]
        found = np.take_along_axis(y_val, indexes, axis=1)
        top_k[f'top_{k}'] = {
            'top_k': k,
            'hit_rate': float(np.mean(found.sum(axis=1) > 0)),
            'label_recall_at_k': float(np.mean(found.sum(axis=1) / y_val.sum(axis=1))),
        }

    metrics = {
        'selected_threshold': best['threshold'],
        'micro_precision': best['micro_precision'],
        'micro_recall': best['micro_recall'],
        'micro_f1': best['micro_f1'],
        'macro_f1_all_classes': float(f1_score(y_val, predicted, average='macro', zero_division=0)),
        'macro_f1_validation_supported_classes': float(f1_score(y_val[:, supported], predicted[:, supported], average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_val, predicted, average='weighted', zero_division=0)),
        'top_k': top_k,
    }
    return metrics, threshold_search

## 8. Train and save V1, V2 and V3

We train the models separately and save each model, its validation JSON report and its training history CSV.

The files are saved directly in **`rnn_selected`**, without version subfolders. This keeps the original RNN experiments safe.

In [ ]:
# Check all output filenames before starting training.
output_files = [REPORT_DIR / 'rnn_comparison.csv', REPORT_DIR / 'rnn_comparison.json']
for version in versions:
    name = f'rnn_{version}'
    output_files += [MODEL_DIR / f'{name}.pt',
                     REPORT_DIR / f'{name}_validation_metrics.json',
                     REPORT_DIR / f'{name}_training_history.csv']
for path in output_files:
    if path.exists():
        raise FileExistsError(f'{path} already exists. Move or rename it before running again.')

results = []
all_history = {}

for version, settings in versions.items():
    name = f'rnn_{version}'
    print(f'\n===== Training {name} =====')
    model, history, best_epoch, seconds = train_model(settings)
    metrics, threshold_search = evaluate_model(model)

    if metrics['micro_f1'] < 0.10 or metrics['top_k']['top_1']['hit_rate'] < 0.10:
        raise RuntimeError(f'{name} scores are unusually low. Check training before saving.')

    metadata = {
        'name': name,
        'type': 'Bidirectional vanilla RNN (multi-label)',
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'training_examples': len(train_rows),
        'validation_examples': len(val_rows),
        'class_count': len(class_names),
        'validation_supported_class_count': int(supported.sum()),
        'validation_missing_class_count': int((~supported).sum()),
        'best_epoch': best_epoch,
        'trained_epochs': len(history),
        'training_seconds': round(seconds, 2),
        'pytorch_version': str(torch.__version__),
        'device_used': DEVICE,
        'random_seed': SEED,
        'dataset_sha256': data_hashes,
        'model_config': {'dropout': settings['dropout']},
        'training': {k: v for k, v in settings.items() if k != 'dropout'},
        'evaluation_scope': 'NHS-derived validation only; test set not accessed',
        'probability_note': 'Scores are not calibrated clinical disease probabilities',
        'demographics_note': 'Age and gender are not trained because NHS data lacks patient-level metadata',
    }

    model_file = MODEL_DIR / f'{name}.pt'
    metrics_file = REPORT_DIR / f'{name}_validation_metrics.json'
    history_file = REPORT_DIR / f'{name}_training_history.csv'
    torch.save({
        'model_state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'model_config': {'dropout': settings['dropout']},
        'vocab': vocab,
        'class_names': class_names,
        'selected_threshold': metrics['selected_threshold'],
        'metadata': metadata,
    }, model_file)
    metrics_file.write_text(json.dumps({
        'model': name, 'metadata': metadata, 'validation_metrics': metrics,
        'threshold_search': threshold_search, 'test_set_used': False,
    }, indent=2), encoding='utf-8')
    history.to_csv(history_file, index=False)

    # Make sure the saved weights can be loaded again.
    saved = torch.load(model_file, map_location='cpu', weights_only=True)
    check_model = SymptomRNN(len(saved['vocab']), len(saved['class_names']),
                             **saved['model_config'])
    check_model.load_state_dict(saved['model_state_dict'])
    assert saved['class_names'] == class_names

    all_history[name] = history
    results.append({
        'model': name,
        'dropout': settings['dropout'],
        'weight_decay': settings['weight_decay'],
        'scheduler': settings['scheduler'],
        'best_epoch': best_epoch,
        'threshold': metrics['selected_threshold'],
        'micro_precision': metrics['micro_precision'],
        'micro_recall': metrics['micro_recall'],
        'micro_f1': metrics['micro_f1'],
        'macro_f1_supported': metrics['macro_f1_validation_supported_classes'],
        'weighted_f1': metrics['weighted_f1'],
        'top_1': metrics['top_k']['top_1']['hit_rate'],
        'top_3': metrics['top_k']['top_3']['hit_rate'],
        'top_5': metrics['top_k']['top_5']['hit_rate'],
        'training_seconds': round(seconds, 2),
    })
    print(f'Saved {name}: Micro F1={metrics["micro_f1"]:.2%}, Top-5={metrics["top_k"]["top_5"]["hit_rate"]:.2%}')
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print('Finished training all three models.')

## 9. Compare our three RNN models

We select the model with the highest validation Micro F1 and also compare the Top-K scores. In a tie, the simpler configuration appears first.

In [ ]:
comparison = pd.DataFrame(results).sort_values(
    ['micro_f1', 'top_1'], ascending=False, kind='stable')
percent_cols = ['micro_precision', 'micro_recall', 'micro_f1',
                'macro_f1_supported', 'weighted_f1', 'top_1', 'top_3', 'top_5']
display(comparison.style.format({col: '{:.2%}' for col in percent_cols}))

comparison.to_csv(REPORT_DIR / 'rnn_comparison.csv', index=False)
(REPORT_DIR / 'rnn_comparison.json').write_text(json.dumps({
    'selected_model': comparison.iloc[0]['model'],
    'selection_metric': 'validation micro_f1, then top_1',
    'dataset_sha256': data_hashes,
    'comparison': results,
    'test_set_used': False,
}, indent=2), encoding='utf-8')

ax = comparison.set_index('model')[['micro_f1', 'top_1', 'top_3', 'top_5']].mul(100).plot.bar(figsize=(8, 4))
ax.set_title('RNN model comparison — validation')
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print('Selected model:', comparison.iloc[0]['model'])

## 10. Plot training results

We check training and validation loss, plus Micro F1 (at threshold 0.5) and Top-5 for each epoch.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, history) in zip(axes, all_history.items()):
    ax.plot(history['epoch'], history['train_loss'], label='Train loss')
    ax.plot(history['epoch'], history['validation_loss'], label='Validation loss')
    ax.set_title(name.upper())
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, history) in zip(axes, all_history.items()):
    ax.plot(history['epoch'], history['validation_f1_at_05'], label='F1 at 0.5')
    ax.plot(history['epoch'], history['validation_top5'], label='Top-5')
    ax.set_title(name.upper())
    ax.set_xlabel('Epoch')
    ax.set_ylim(0, 1)
    ax.legend()
plt.tight_layout()
plt.show()

## 11. Compare with our Logistic Regression baseline

These are the validation results from the Logistic Regression experiment. We include them for reference only; we are not retraining Logistic Regression here.

In [ ]:
baseline = pd.DataFrame([
    {'model': 'Logistic Regression V3', 'micro_f1': 0.783640081799591,
     'top_1': 0.887000849617672, 'top_3': 0.9498725573491928,
     'top_5': 0.9694137638062872},
    {'model': 'Logistic Regression V2 (best Top-K)', 'micro_f1': 0.780448065173116,
     'top_1': 0.897196261682243, 'top_3': 0.9575191163976211,
     'top_5': 0.9753610875106202},
])
summary = pd.concat([baseline, comparison[['model', 'micro_f1', 'top_1', 'top_3', 'top_5']]],
                    ignore_index=True)
display(summary.style.format({col: '{:.2%}' for col in ('micro_f1', 'top_1', 'top_3', 'top_5')}))
print('This comparison uses validation data only.')

## 12. Try a symptom example

Finally, we load the selected model from disk and see its five highest-ranked conditions. This is a technical demonstration, not medical advice.

In [ ]:
best_name = comparison.iloc[0]['model']
saved = torch.load(MODEL_DIR / f'{best_name}.pt', map_location=DEVICE, weights_only=True)
best_model = SymptomRNN(len(saved['vocab']), len(saved['class_names']),
                        **saved['model_config']).to(DEVICE)
best_model.load_state_dict(saved['model_state_dict'])
best_model.eval()

sample_text = 'persistent cough wheezing and shortness of breath'
x = torch.tensor([encode(sample_text)], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    scores = torch.sigmoid(best_model(x))[0].cpu().numpy()

indexes = np.argsort(-scores)[:5]
display(pd.DataFrame({
    'condition': np.array(saved['class_names'])[indexes],
    'model_score': np.round(scores[indexes], 4),
    'above_threshold': scores[indexes] >= saved['selected_threshold'],
}))

## Conclusion

We trained and compared three bidirectional RNN configurations using the same dataset. Our results, models and graphs are now saved.

We can select the strongest RNN and compare it fairly with our next models, **LSTM** and **BioBERT**. The final test set remains untouched, and the validation results do not represent clinical diagnostic accuracy.